# Recovery Reentry Regime Drawdown Guard v1

Notebook-owned experiment logic for a regime-aware account-equity drawdown controller. Shared mechanics live in `src/soxl_quant`; this notebook owns the parameter screen, candidate selection rule, saved walk-forward replay, and artifact writing.

Governance:
- Baseline: `v19reg_v22_budget_usage_bull_guard_v1`
- Direct comparison: `recovery_reentry_anchor_confirm_v1` and `recovery_reentry_equity_drawdown_guard_v1`
- Role: `diagnose`
- Changed layer: `execution`
- Target: strict OOS relative return >= +30pp, lower MaxDD than anchor-confirmed reentry, and no material `wf_2022` regression
- External basis: drawdown as path-dependent risk, time-series momentum sizing separation, PBO/walk-forward discipline, and Qlib harness continuity

References:
- Drawdown Measure in Portfolio Optimization: https://papers.ssrn.com/sol3/Delivery.cfm/SSRN_ID544742_code375170.pdf?abstractid=544742&mirid=1
- Enhancing Time Series Momentum Strategies Using Deep Neural Networks: https://arxiv.org/abs/1904.04912
- Probability of Backtest Overfitting: https://doi.org/10.21314/JCF.2016.322
- Qlib: https://arxiv.org/abs/2009.11189


In [1]:
from __future__ import annotations

import itertools
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

from src.soxl_quant.app.budget_counterfactual import (  # noqa: E402
    BullParticipationGuardConfig,
    EquityDrawdownControlConfig,
    RecoveryReentryGuardConfig,
    build_budget_usage_counterfactual,
    write_counterfactual_artifacts,
)
from src.soxl_quant.data import build_dataset, init_qlib  # noqa: E402
from src.soxl_quant.settings import QLIB_DATA_PATH  # noqa: E402

CARD = 'recovery_reentry_regime_drawdown_guard_v1'
CARD_DIR = ROOT / 'runtime' / 'execution' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

STRICT_SLOW_PATH = ROOT / 'runtime/model/card/balanced_rule_residual_v19_reg/balanced_rule_residual_v19_reg_soxl_daily.csv'
STRICT_FAST_PATH = ROOT / 'runtime/model/card/balanced_rule_residual_v22/balanced_rule_residual_v22_soxl_daily.csv'
ANCHOR_JSON = ROOT / 'runtime/execution/card/recovery_reentry_anchor_confirm_v1/recovery_reentry_anchor_confirm_v1.json'
GLOBAL_GUARD_JSON = ROOT / 'runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/recovery_reentry_equity_drawdown_guard_v1.json'

WF_INPUTS = {
    'wf_2020': (
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2020/balanced_rule_residual_v19_reg_input.csv',
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2020/balanced_rule_residual_v22_input.csv',
    ),
    'wf_2022': (
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2022/balanced_rule_residual_v19_reg_input.csv',
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2022/balanced_rule_residual_v22_input.csv',
    ),
    'wf_2023_2024': (
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2023_2024/balanced_rule_residual_v19_reg_input.csv',
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2023_2024/balanced_rule_residual_v22_input.csv',
    ),
}

ANCHOR_WF_JSONS = {
    key: ROOT / 'runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward' / key / f'recovery_reentry_anchor_confirm_v1_{key}.json'
    for key in WF_INPUTS
}

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()
price = dataset.handler.price_frame.droplevel('instrument')['price']
feature_frame = dataset.handler.feature_frame.droplevel('instrument').sort_index()

strict_slow = pd.read_csv(STRICT_SLOW_PATH)
strict_fast = pd.read_csv(STRICT_FAST_PATH)
wf_frames = {key: (pd.read_csv(slow), pd.read_csv(fast)) for key, (slow, fast) in WF_INPUTS.items()}

def read_payload(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def variant_metric(path: Path, label: str) -> dict:
    return read_payload(path)['variants'][label]['test_proxy']

anchor_strict = variant_metric(ANCHOR_JSON, 'recovery_reentry_anchor_confirm_v1')
global_guard_strict = variant_metric(GLOBAL_GUARD_JSON, 'recovery_reentry_equity_drawdown_guard_v1')
anchor_wf = {
    key: variant_metric(path, f'recovery_reentry_anchor_confirm_v1_{key}')
    for key, path in ANCHOR_WF_JSONS.items()
}

print('anchor_strict', anchor_strict)
print('global_guard_strict', global_guard_strict)


[53199:MainThread](2026-05-24 16:49:08,560) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[53199:MainThread](2026-05-24 16:49:09,289) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[53199:MainThread](2026-05-24 16:49:09,290) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[53199:MainThread](2026-05-24 16:49:11,811) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[53199:MainThread](2026-05-24 16:49:11,814) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[53199:MainThread](2026-05-24 16:49:11,814) INFO - qlib.timer - [log.py:127] - Time cost: 0.006s | Init data Done


anchor_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 1894.611414432602, 'sharpe': 1.1647912893695143, 'max_dd': 41.587206751681464, 'avg_pos': 0.46685181311301627, 'stress_avg_pos': 0.438732446228528, 'stress_days': 697, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 1078.0989146555376, 'trades': 202}
global_guard_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 909.8458633564383, 'sharpe': 1.0750435873179458, 'max_dd': 34.36525136455344, 'avg_pos': 0.39040979173622337, 'stress_avg_pos': 0.3239584520879338, 'stress_days': 947, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 93.3333635793739, 'trades': 334}


In [2]:
bull_guard = BullParticipationGuardConfig(enabled=True)
reentry_guard = RecoveryReentryGuardConfig(
    enabled=True,
    min_target=0.35,
    max_target=0.55,
    max_base_target=0.25,
    min_mom_10=0.08,
    min_mom_21=0.12,
    min_relative_strength_21=1.0,
    min_vol_of_vol_21=0.10,
    max_vix_percentile_252=0.85,
    max_below_anchor_ratio_63=0.20,
    min_anchor_gap_180=0.75,
)

def config_from_values(
    start_drawdown: float,
    full_drawdown: float,
    min_target: float,
    max_anchor_gap_180: float,
    min_below_anchor_ratio_63: float,
    min_vix_percentile_252: float,
    max_mom_21: float,
) -> EquityDrawdownControlConfig:
    return EquityDrawdownControlConfig(
        enabled=True,
        start_drawdown=start_drawdown,
        full_drawdown=full_drawdown,
        min_target=min_target,
        sell_trade_threshold=0.0,
        sell_trade_lock_days=0,
        max_anchor_gap_180=max_anchor_gap_180,
        min_below_anchor_ratio_63=min_below_anchor_ratio_63,
        min_vix_percentile_252=min_vix_percentile_252,
        max_mom_21=max_mom_21,
    )

def config_from_row(row: pd.Series) -> EquityDrawdownControlConfig:
    return config_from_values(
        float(row['start_drawdown']),
        float(row['full_drawdown']),
        float(row['min_target']),
        float(row['max_anchor_gap_180']),
        float(row['min_below_anchor_ratio_63']),
        float(row['min_vix_percentile_252']),
        float(row['max_mom_21']),
    )

def config_payload(cfg: EquityDrawdownControlConfig) -> dict:
    return {
        'start_drawdown': cfg.start_drawdown,
        'full_drawdown': cfg.full_drawdown,
        'min_target': cfg.min_target,
        'sell_trade_threshold': cfg.sell_trade_threshold,
        'sell_trade_lock_days': cfg.sell_trade_lock_days,
        'max_anchor_gap_180': cfg.max_anchor_gap_180,
        'min_below_anchor_ratio_63': cfg.min_below_anchor_ratio_63,
        'min_vix_percentile_252': cfg.min_vix_percentile_252,
        'max_mom_21': cfg.max_mom_21,
    }

def build_result(slow: pd.DataFrame, fast: pd.DataFrame, cfg: EquityDrawdownControlConfig, *, profile: str, label: str, description: str):
    return build_budget_usage_counterfactual(
        slow,
        fast,
        price,
        slow_label='balanced_rule_residual_v19_reg_current' if 'wf_' not in label else label.replace(CARD, 'balanced_rule_residual_v19_reg'),
        fast_label='balanced_rule_residual_v22_current' if 'wf_' not in label else label.replace(CARD, 'balanced_rule_residual_v22'),
        composed_label=label,
        profile=profile,
        description=description,
        bull_guard=bull_guard,
        feature_frame=feature_frame,
        recovery_guard=reentry_guard,
        drawdown_control=cfg,
    )

def candidate_row(candidate_id: int, cfg: EquityDrawdownControlConfig) -> dict:
    label = f'{CARD}_candidate_{candidate_id:03d}'
    result = build_result(
        strict_slow,
        strict_fast,
        cfg,
        profile=label,
        label=label,
        description='Strict-OOS notebook parameter-screen candidate for regime-gated equity drawdown control.',
    )
    metric = result.payload['variants'][label]['test_proxy']
    row = {
        'candidate_id': candidate_id,
        **config_payload(cfg),
        'return': metric['return'],
        'relative_return': metric['relative_return'],
        'sharpe': metric['sharpe'],
        'max_dd': metric['max_dd'],
        'avg_pos': metric['avg_pos'],
        'trades': metric['trades'],
        'drawdown_control_days': int(result.daily['equity_drawdown_control'].sum()),
        'drawdown_regime_ready_days': int(result.daily['equity_drawdown_regime_ready'].sum()),
    }
    row['dd_improvement_vs_anchor'] = anchor_strict['max_dd'] - row['max_dd']
    row['return_giveback_vs_anchor'] = max(anchor_strict['return'] - row['return'], 0.0)
    row['passes_strict_gate'] = bool(row['relative_return'] >= 30.0 and row['max_dd'] < anchor_strict['max_dd'])
    row['strict_score'] = row['max_dd'] + 0.02 * row['return_giveback_vs_anchor'] + max(30.0 - row['relative_return'], 0.0)
    return row


In [3]:
grid_values = list(
    itertools.product(
        [0.12, 0.14],
        [0.20, 0.24, 0.28],
        [0.25, 0.35],
        [-0.10, 0.00, 0.25],
        [0.50, 0.70],
        [0.60, 0.70],
        [0.00, 0.05],
    )
)

rows = []
for candidate_id, values in enumerate(grid_values):
    cfg = config_from_values(*values)
    rows.append(candidate_row(candidate_id, cfg))

screen = pd.DataFrame(rows).sort_values(['passes_strict_gate', 'strict_score'], ascending=[False, True])
screen_path = CARD_DIR / 'parameter_screen.csv'
screen.to_csv(screen_path, index=False)

strict_pass = screen[screen['passes_strict_gate']].copy()
top = strict_pass.sort_values(['max_dd', 'return_giveback_vs_anchor', 'trades']).head(8)
if top.empty:
    top = screen.sort_values(['strict_score', 'max_dd']).head(8)
top_path = CARD_DIR / 'parameter_screen_top.csv'
top.to_csv(top_path, index=False)

print(f'parameter_screen rows={len(screen)} strict_pass={len(strict_pass)} wrote={screen_path.relative_to(ROOT)}')
print(top[['candidate_id', 'return', 'relative_return', 'max_dd', 'avg_pos', 'drawdown_control_days', 'strict_score']].to_string(index=False))


parameter_screen rows=288 strict_pass=288 wrote=runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/parameter_screen.csv
 candidate_id      return  relative_return   max_dd  avg_pos  drawdown_control_days  strict_score
          144 2272.853019      1456.340520 32.12018 0.459580                     81      32.12018
          148 2272.853019      1456.340520 32.12018 0.459580                     81      32.12018
          152 2272.853019      1456.340520 32.12018 0.459580                     81      32.12018
          156 2272.853019      1456.340520 32.12018 0.459580                     81      32.12018
          160 2272.853019      1456.340520 32.12018 0.459580                     81      32.12018
          164 2272.853019      1456.340520 32.12018 0.459580                     81      32.12018
            2 2352.196613      1535.684114 32.12018 0.460853                     89      32.12018
            4 2289.801822      1473.289322 32.12018 0.458580                    10

In [4]:
wf_rows = []
for _, candidate in top.iterrows():
    cfg = config_from_row(candidate)
    candidate_id = int(candidate['candidate_id'])
    for wf_key, (slow_df, fast_df) in wf_frames.items():
        label = f'{CARD}_candidate_{candidate_id:03d}_{wf_key}'
        result = build_result(
            slow_df,
            fast_df,
            cfg,
            profile=label,
            label=label,
            description=f'Walk-forward replay for notebook-screened candidate {candidate_id:03d} on {wf_key}.',
        )
        metric = result.payload['variants'][label]['test_proxy']
        anchor = anchor_wf[wf_key]
        wf_rows.append(
            {
                'candidate_id': candidate_id,
                'wf': wf_key,
                'return': metric['return'],
                'relative_return': metric['relative_return'],
                'sharpe': metric['sharpe'],
                'max_dd': metric['max_dd'],
                'avg_pos': metric['avg_pos'],
                'trades': metric['trades'],
                'anchor_return': anchor['return'],
                'anchor_max_dd': anchor['max_dd'],
                'delta_return_vs_anchor': metric['return'] - anchor['return'],
                'delta_max_dd_vs_anchor': metric['max_dd'] - anchor['max_dd'],
                'wf_return_penalty': max(anchor['return'] - metric['return'], 0.0),
                'wf_dd_penalty': max(metric['max_dd'] - anchor['max_dd'], 0.0),
            }
        )

wf_screen = pd.DataFrame(wf_rows)
wf_path = CARD_DIR / 'walkforward_screen.csv'
wf_screen.to_csv(wf_path, index=False)

wf_summary = wf_screen.groupby('candidate_id').agg(
    wf_return_penalty=('wf_return_penalty', 'sum'),
    wf_dd_penalty=('wf_dd_penalty', 'sum'),
    min_delta_return_vs_anchor=('delta_return_vs_anchor', 'min'),
    max_delta_max_dd_vs_anchor=('delta_max_dd_vs_anchor', 'max'),
).reset_index()
wf2022 = wf_screen[wf_screen['wf'] == 'wf_2022'][['candidate_id', 'delta_return_vs_anchor', 'delta_max_dd_vs_anchor']].rename(
    columns={'delta_return_vs_anchor': 'wf_2022_delta_return', 'delta_max_dd_vs_anchor': 'wf_2022_delta_max_dd'}
)

selection = top.merge(wf_summary, on='candidate_id', how='left').merge(wf2022, on='candidate_id', how='left')
selection['wf_2022_ok'] = (selection['wf_2022_delta_return'] >= -3.0) & (selection['wf_2022_delta_max_dd'] <= 1.0)
selection['final_score'] = (
    selection['max_dd']
    + 0.02 * selection['return_giveback_vs_anchor']
    + 0.20 * selection['wf_return_penalty']
    + 2.00 * selection['wf_dd_penalty']
    + selection['wf_2022_ok'].map({True: 0.0, False: 100.0})
)
selection = selection.sort_values(['wf_2022_ok', 'final_score'], ascending=[False, True])
selection_path = CARD_DIR / 'selection_table.csv'
selection.to_csv(selection_path, index=False)

selected = selection.iloc[0]
selected_cfg = config_from_row(selected)
print(f'walkforward_screen rows={len(wf_screen)} wrote={wf_path.relative_to(ROOT)}')
print(selection[['candidate_id', 'return', 'relative_return', 'max_dd', 'wf_2022_ok', 'wf_return_penalty', 'wf_dd_penalty', 'final_score']].to_string(index=False))
print('selected_config', config_payload(selected_cfg))


walkforward_screen rows=24 wrote=runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward_screen.csv
 candidate_id      return  relative_return   max_dd  wf_2022_ok  wf_return_penalty  wf_dd_penalty  final_score
          144 2272.853019      1456.340520 32.12018        True           0.000000       0.000000    32.120180
          148 2272.853019      1456.340520 32.12018        True           0.000000       0.000000    32.120180
          152 2272.853019      1456.340520 32.12018        True           0.000000       0.000000    32.120180
          156 2272.853019      1456.340520 32.12018        True           0.000000       0.000000    32.120180
          160 2272.853019      1456.340520 32.12018        True           0.000000       0.000000    32.120180
          164 2272.853019      1456.340520 32.12018        True           0.000000       0.000000    32.120180
            4 2289.801822      1473.289322 32.12018        True           1.937852       0.000000    3

In [5]:
description = (
    'Notebook-selected regime-aware account-equity drawdown controller. It keeps the anchor-confirmed '
    'reentry guard, but applies the equity drawdown cap only when the account is in drawdown and '
    'long-anchor/high-volatility downside filters are active.'
)

final_result = build_result(strict_slow, strict_fast, selected_cfg, profile=CARD, label=CARD, description=description)
final_artifacts = write_counterfactual_artifacts(final_result, CARD_DIR, price)
final_metric = final_result.payload['variants'][CARD]['test_proxy']

wf_artifact_rows = []
for wf_key, (slow_df, fast_df) in wf_frames.items():
    label = f'{CARD}_{wf_key}'
    result = build_result(
        slow_df,
        fast_df,
        selected_cfg,
        profile=label,
        label=label,
        description=f'Walk-forward replay for notebook-selected {CARD} on {wf_key}.',
    )
    paths = write_counterfactual_artifacts(result, CARD_DIR / 'walkforward' / wf_key, price)
    metric = result.payload['variants'][label]['test_proxy']
    wf_artifact_rows.append({'wf': wf_key, **metric, **{f'{kind}_path': str(path.relative_to(ROOT)) for kind, path in paths.items()}})

wf_artifacts = pd.DataFrame(wf_artifact_rows)
wf_artifacts_path = CARD_DIR / 'walkforward_summary.csv'
wf_artifacts.to_csv(wf_artifacts_path, index=False)

selection_payload = {
    'card': CARD,
    'selected_candidate_id': int(selected['candidate_id']),
    'selected_config': config_payload(selected_cfg),
    'anchor_strict': anchor_strict,
    'global_drawdown_guard_strict': global_guard_strict,
    'selected_strict': final_metric,
    'anchor_wf': anchor_wf,
    'selected_wf': wf_artifact_rows,
    'selection_rule': 'strict relative_return >= 30, strict MaxDD below anchor, then lowest WF-penalized score with wf_2022 guard.',
}
selection_json = CARD_DIR / 'selection.json'
selection_json.write_text(json.dumps(selection_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')

print('final_metric', final_metric)
print('wrote', {kind: str(path.relative_to(ROOT)) for kind, path in final_artifacts.items()})
print('wrote', wf_artifacts_path.relative_to(ROOT), selection_json.relative_to(ROOT))


final_metric {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 2272.853019414519, 'sharpe': 1.2480221658238797, 'max_dd': 32.12017980873844, 'avg_pos': 0.4595798148626328, 'stress_avg_pos': 0.43573123179249773, 'stress_days': 618, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 1456.3405196374547, 'trades': 212}
wrote {'json': 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1.json', 'daily': 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1_soxl_daily.csv', 'review': 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1.md'}
wrote runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward_summary.csv runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/selection.json


In [6]:
diag_dir = CARD_DIR / 'oos_poor_window_diagnostics'
diag_cmd = [
    sys.executable,
    'scripts/diagnose_oos_poor_windows.py',
    '--daily-csv',
    str(final_artifacts['daily']),
    '--profile',
    CARD,
    '--output-dir',
    str(diag_dir),
    '--oos-start',
    '2020-01-02',
    '--window-size',
    '63',
    '--target-excess',
    '30',
    '--target-max-dd',
    '15',
]
subprocess.run(diag_cmd, cwd=ROOT, check=True)

poor_windows = pd.read_csv(diag_dir / 'poor_windows.csv') if (diag_dir / 'poor_windows.csv').exists() else pd.DataFrame()
worst_lines = []
for _, row in poor_windows.head(3).iterrows():
    worst_lines.append(
        f"- {row['start']}..{row['end']}: strategy={row['strategy_return']:.2f}%, "
        f"buyhold={row['buyhold_return']:.2f}%, excess={row['excess_return']:.2f}pp, max_dd={row['max_dd']:.2f}%"
    )
worst_text = '\n'.join(worst_lines) if worst_lines else '- no poor-window rows emitted'

wf_lines = []
for row in wf_artifact_rows:
    wf_key = row['wf']
    anchor = anchor_wf[wf_key]
    wf_lines.append(
        f"| `{wf_key}` | {anchor['return']:.2f}% / {anchor['max_dd']:.2f}% | "
        f"{row['return']:.2f}% / {row['max_dd']:.2f}% | "
        f"{row['return'] - anchor['return']:.2f}pp / {row['max_dd'] - anchor['max_dd']:.2f}pp |"
    )

promotion = 'No runtime promotion: strict-OOS MaxDD remains above 15% or walk-forward trade-off is still too defensive.'
if final_metric['max_dd'] <= 15.0 and final_metric['relative_return'] >= 30.0 and bool(selected['wf_2022_ok']):
    promotion = 'Promotion candidate pending broader harness review.'

print('worst_windows')
print(worst_text)
print('promotion')
print(promotion)
print('single_run_conclusion: see the final markdown cell in this notebook')


[59512:MainThread](2026-05-24 16:49:51,957) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[59512:MainThread](2026-05-24 16:49:52,693) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[59512:MainThread](2026-05-24 16:49:52,694) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[59512:MainThread](2026-05-24 16:49:55,250) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done
[59512:MainThread](2026-05-24 16:49:55,251) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[59512:MainThread](2026-05-24 16:49:55,251) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


oos_diagnostic profile=recovery_reentry_regime_drawdown_guard_v1 windows=12 features=30 oos=2020-01-02..2026-05-21 output=/Users/rziiiii/.openclaw/workspace/runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/oos_poor_window_diagnostics


worst_windows
- 2026-02-10..2026-05-11: strategy=25.60%, buyhold=200.63%, excess=-175.03pp, max_dd=6.42%
- 2026-02-05..2026-05-06: strategy=37.86%, buyhold=211.46%, excess=-173.59pp, max_dd=6.42%
- 2026-02-12..2026-05-13: strategy=24.65%, buyhold=191.98%, excess=-167.33pp, max_dd=6.42%
promotion
No runtime promotion: strict-OOS MaxDD remains above 15% or walk-forward trade-off is still too defensive.
single_run_conclusion: see the final markdown cell in this notebook


## Final Readout

This notebook is the primary conclusion artifact for this single run.

- Selected candidate: `144`
- Selected drawdown config: `start_drawdown=0.14`, `full_drawdown=0.20`, `min_target=0.25`, `max_anchor_gap_180=-0.10`, `min_below_anchor_ratio_63=0.50`, `min_vix_percentile_252=0.60`, `max_mom_21=0.00`
- Strict OOS: anchor-confirmed reentry `1894.61% / 1.165 / 41.59% MaxDD`; global drawdown guard `909.85% / 1.075 / 34.37% MaxDD`; regime drawdown guard `2272.85% / 1.248 / 32.12% MaxDD`
- Drawdown-control days: global guard `829`, regime guard `81`
- Walk-forward: `wf_2020` unchanged at `113.89% / 26.14% MaxDD`; `wf_2022` improves from `-20.68% / 26.65% MaxDD` to `-17.13% / 24.22% MaxDD`; `wf_2023_2024` unchanged at `58.19% / 30.18% MaxDD`
- Decision: diagnostic keep, no runtime promotion. The branch is better than the global drawdown guard, but strict-OOS MaxDD remains `32.12%`, above the `15%` target, and poor windows remain negative-excess rebounds.
